# Notebook 09: Training Loop (train.py)

**Module:** 12 Capstone — water-bodies-detection  
**Duration:** ~3 hrs

---

## Learning Objectives

1. Walk main() from config load to checkpoint save
2. Understand train_epoch with AMP
3. Trace two-stage freeze/unfreeze
4. Debug a stalled training run

## train.py Flow

```
1. Load config/default.yaml
2. get_stem_triplets() — match tiles + masks by filename
3. train_test_split (tile-level, 85/15)
4. WaterBodyTileDataset + DataLoader
5. build_water_model()
6. Stage 1: freeze encoder, train decoder (12 epochs)
7. Stage 2: unfreeze all, early stopping (120 max)
8. Save best.pt, final.pt, model_meta.json
```

In [ ]:
import os, json, yaml
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (8, 5)
REPO = '../../water-bodies-detection'
print('Capstone repo path:', os.path.abspath(REPO) if os.path.exists(REPO) else 'clone water-bodies-detection alongside Machine-Learning')

## train_epoch() — Line by Line

```python
for x, y in loader:
    x, y = x.to(device), y.to(device)
    with torch.amp.autocast('cuda'):      # FP16 forward
        logits = model(x)                 # (B, 2, H, W)
    logits = logits.float()               # FP32 loss
    loss = criterion(logits, y)           # AquaBoundaryLoss
    scaler.scale(loss).backward()
    clip_grad_norm_(..., grad_clip_norm)  # max norm 1.0
    scaler.step(optimizer); scaler.update()
```

**AMP:** Forward in FP16, loss in FP32 — prevents numerical issues.

## Two-Stage Training

**Stage 1 (epochs 1–12):**
- `set_encoder_trainable(model, False)`
- LR = 2e-4, warmup 2 epochs
- Decoder learns on frozen ImageNet features

**Stage 2 (epochs 1–120):**
- `set_encoder_trainable(model, True)`
- LR = 2e-5 (10× lower)
- Full fine-tune with early stopping (patience=18 on val aqua IoU)

---

## Summary

train.py orchestrates everything — know it line by line.

**Next:** [10_Optimizer_and_Scheduler.ipynb](10_Optimizer_and_Scheduler.ipynb)